# Phase 1 — SEC Bitcoin Press Release Web Scraping

### Goal

Build a polite and reproducible acquisition pipeline that discovers SEC crypto press releases, parses complete article pages, filters Bitcoin-related documents, validates the collection, and saves CSV/Parquet/JSONL outputs.


## 1. Why SEC?

SEC press releases are official, timestamped financial-regulatory documents. They provide a useful event-driven source for later Financial NLP experiments involving Bitcoin regulation, enforcement, exchanges, mining, and ETPs.


## 2. Access policy

The scraper identifies itself with a declared User-Agent, limits requests to one per second by default, uses an SEC-only host allow-list, and stops on access blocking or rate limiting. Do not increase the request rate unnecessarily.


In [1]:
from pathlib import Path
import sys
import json

import pandas as pd

SCRAPER_ROOT = Path.cwd().resolve()
if not (SCRAPER_ROOT / "configs" / "scraper_config.yaml").is_file():
    for candidate in [SCRAPER_ROOT, *SCRAPER_ROOT.parents]:
        if (candidate / "configs" / "scraper_config.yaml").is_file():
            SCRAPER_ROOT = candidate
            break

if str(SCRAPER_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRAPER_ROOT))

from src.config import load_config, resolve_config_paths
from src.pipeline import import_verified_snapshot, run_live_scrape

CONFIG_PATH = SCRAPER_ROOT / "configs" / "scraper_config.yaml"
config = resolve_config_paths(load_config(CONFIG_PATH), SCRAPER_ROOT)
print("Scraper root:", SCRAPER_ROOT)
print("Source:", config["source"]["base_url"])


Scraper root: C:\Users\sepehr\PycharmProjects\FinancialNLP\1_data_acquisition\live\news\scraper\sec_crypto_news
Source: https://www.sec.gov


## 3. Inspect the configuration

The discovery query is `crypto`, while the final document filter searches title, description, and full body for Bitcoin-specific terms. This avoids missing releases whose titles are broad but whose body discusses Bitcoin.


In [2]:
config["source"], config["bitcoin_filter"], config["http"]


({'name': 'sec.gov',
  'source_type': 'press_release',
  'base_url': 'https://www.sec.gov',
  'listing_path': '/newsroom/press-releases',
  'search_query': 'crypto',
  'start_page': 0,
  'max_pages': 6,
  'allowed_hosts': ['www.sec.gov', 'sec.gov']},
 {'case_sensitive': False,
  'keywords': ['bitcoin',
   'bitcoins',
   'btc',
   'spot bitcoin',
   'bitcoin etf',
   'bitcoin etp',
   'bitcoin exchange',
   'bitcoin mining'],
  'required_matches': 1},
 {'user_agent': 'FinancialNLPForCrypto/1.0 contact=sepehrpg@users.noreply.github.com',
  'accept': 'text/html,application/xhtml+xml',
  'accept_encoding': 'gzip, deflate',
  'timeout_seconds': 30,
  'delay_seconds': 1.0,
  'max_retries': 3,
  'backoff_factor': 2.0,
  'verify_ssl': True})

## 4. Choose execution mode

The included verified snapshot is safe for a reproducible classroom demonstration. Set `RUN_LIVE = True` to contact SEC.gov and collect complete current pages.


In [3]:
RUN_LIVE = False
MAX_PAGES = 2
MAX_ARTICLES = 40

if RUN_LIVE:
    frame = run_live_scrape(
        config,
        max_pages=MAX_PAGES,
        max_articles=MAX_ARTICLES,
        incremental=True,
    )
else:
    snapshot_path = SCRAPER_ROOT / "data" / "snapshots" / "verified_bitcoin_press_releases.jsonl"
    frame = import_verified_snapshot(config, snapshot_path)

print("Rows:", len(frame))
print("Date range:", frame["published_at"].min(), "to", frame["published_at"].max())


Rows: 16
Date range: 2015-12-01 to 2025-09-17


## 5. Inspect the collected documents


In [4]:
display_columns = [
    "release_number", "published_at", "title", "bitcoin_keyword_count",
    "word_count", "body_text_status"
]
frame[display_columns].head(10)


,release_number,published_at,title,bitcoin_keyword_count,word_count,body_text_status
0,2025-121,2025-09-17,SEC Approves Generic Listing Standards for Com...,2,214,verified_excerpt
1,2025-101,2025-07-29,SEC Permits In-Kind Creations and Redemptions ...,9,138,verified_excerpt
2,2024-125,2024-09-12,eToro Reaches Settlement with SEC and Will Cea...,2,134,verified_excerpt
3,2024-13,2024-02-02,SEC Charges Founder of American Bitcoin Academ...,4,130,verified_excerpt
4,2024-11,2024-01-29,SEC Charges Founder of $1.7 Billion HyperFund ...,2,106,verified_excerpt
5,2022-201,2022-11-04,SEC Charges Creator of Global Crypto Ponzi Sch...,3,125,verified_excerpt
6,2022-81,2022-05-06,SEC Halts Fraudulent Cryptomining and Trading ...,1,94,verified_excerpt
7,2021-237,2021-11-18,SEC Charges Promoter with Conducting Cryptocur...,1,82,verified_excerpt
8,2021-51,2021-03-18,SEC Charges California-Based Fraudster With Se...,1,92,verified_excerpt
9,2021-22,2021-02-01,SEC Charges Three Individuals in Digital Asset...,4,92,verified_excerpt


## 6. Validate output integrity

The final dataset must have unique IDs, release numbers, canonical URLs, and content hashes. Missing titles or text are blocking errors.


In [5]:
from src.validation import quality_report, validate_frame

validate_frame(frame)
quality = quality_report(frame)
quality


,metric,value
0,row_count,16
1,unique_release_numbers,16
2,duplicate_source_row_ids,0
3,duplicate_canonical_urls,0
4,duplicate_content_hashes,0
5,missing_titles,0
6,missing_full_text,0
7,invalid_published_dates,0
8,median_word_count,100.5
9,min_word_count,63


## 7. Main CSV output

The CSV is encoded with UTF-8 BOM for Excel compatibility. Parquet is the preferred downstream format after installing `pyarrow`.


In [6]:
csv_path = Path(config["storage"]["processed_csv"])
print(csv_path)
print("Exists:", csv_path.is_file())
pd.read_csv(csv_path).head(3)


C:\Users\sepehr\PycharmProjects\FinancialNLP\1_data_acquisition\live\news\scraper\sec_crypto_news\data\processed\sec_bitcoin_press_releases.csv
Exists: True


,source_row_id,document_id,source,source_type,release_number,title,description,full_text,published_at,last_reviewed_at,...,full_text_is_complete,source_verified,scraped_at,content_hash,http_status,bitcoin_keyword_count,matched_keywords,word_count,character_count,is_bitcoin_related
0,sec_2025_121,sec_2025_121,sec.gov,press_release,2025-121,SEC Approves Generic Listing Standards for Com...,NaN,The Securities and Exchange Commission today v...,2025-09-17,2025-09-18,...,False,True,2026-08-03T13:03:00+00:00,952b3e4b39dec25e3ba716c60ee918c6933f55b8c2a524...,200,2,bitcoin,214,1477,True
1,sec_2025_101,sec_2025_101,sec.gov,press_release,2025-101,SEC Permits In-Kind Creations and Redemptions ...,NaN,The Securities and Exchange Commission today v...,2025-07-29,2025-07-30,...,False,True,2026-08-03T13:03:00+00:00,3a4ed4d47af23094bbef45dc1e82d02a7d4930097a89e4...,200,9,bitcoin|btc|spot bitcoin,138,900,True
2,sec_2024_125,sec_2024_125,sec.gov,press_release,2024-125,eToro Reaches Settlement with SEC and Will Cea...,Firm charged with operating unregistered broke...,The Securities and Exchange Commission announc...,2024-09-12,2024-09-12,...,False,True,2026-08-03T13:03:00+00:00,8af509b43e0b0db71d9da31cbafb409bd974e293ac7780...,200,2,bitcoin,134,861,True
